# ML Assignment 2: Breast Cancer Classification




In [44]:
# Import the Python libraries we need.
import pandas as pd
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, matthews_corrcoef, confusion_matrix


In [45]:
# Read the dataset from the CSV file in this project folder.
# This CSV has 569 rows and 30 input features.
df = pd.read_csv("breast_cancer_wisconsin_diagonistic.csv")

X=df.drop(columns=['ID','Diagnosis'])
y=(df['Diagnosis']=='M').astype(int).values

# Handle missing values if any
print('\nMissing values in data:')
print(df.isnull().sum())
## No missing data found in the dataset

print("Number of rows:", len(X))
print("Number of features:", X.shape[1])
print("Diagnosis counts:")
print(df["Diagnosis"].value_counts())



Missing values in data:
ID                    0
radius1               0
texture1              0
perimeter1            0
area1                 0
smoothness1           0
compactness1          0
concavity1            0
concave_points1       0
symmetry1             0
fractal_dimension1    0
radius2               0
texture2              0
perimeter2            0
area2                 0
smoothness2           0
compactness2          0
concavity2            0
concave_points2       0
symmetry2             0
fractal_dimension2    0
radius3               0
texture3              0
perimeter3            0
area3                 0
smoothness3           0
compactness3          0
concavity3            0
concave_points3       0
symmetry3             0
fractal_dimension3    0
Diagnosis             0
dtype: int64
Number of rows: 569
Number of features: 30
Diagnosis counts:
Diagnosis
B    357
M    212
Name: count, dtype: int64


In [46]:
# Split into 80% training and 20% test data.
# stratify=y keeps the class proportions similar in both parts.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)

# Save the test data with the target column for the Streamlit upload.
test_data = X_test.copy()
test_data["target"] = y_test
test_data.to_csv("test_data.csv", index=False)
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


Training rows: 455
Testing rows: 114


In [47]:
# Create all five models.
models = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "KNN": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=3)),
    "Naive Bayes": GaussianNB(),
    #"Support Vector Machine": make_pipeline(StandardScaler(), SVC(probability=True, random_state=42))
    "Support Vector Machine": make_pipeline(StandardScaler(), SVC(kernel="rbf", random_state=42))
}

# The saved file name for each model.
file_names = {
    "Logistic Regression": "lr.joblib",
    "Decision Tree": "dt.joblib",
    "KNN": "knn.joblib",
    "Naive Bayes": "nb.joblib",
    "Support Vector Machine": "svm.joblib"
}


In [48]:
# Make a folder to save trained models.
Path("model").mkdir(exist_ok=True)
joblib.dump(list(X.columns), "model/feature_names.joblib")

results = []

for name, model in models.items():
    model.fit(X_train, y_train)  # Train only using training data.
    predictions = model.predict(X_test)  # Predict on unseen test data.
    # Calculate scores for AUC
    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)[:, 1]
    else:
        y_score = model.decision_function(X_test)

    # Calculate the six metrics required by the assignment.
    accuracy = accuracy_score(y_test, predictions)
    auc = roc_auc_score(y_test, y_score)
    precision = precision_score(y_test, predictions)
    recall = recall_score(y_test, predictions)
    f1 = f1_score(y_test, predictions)
    mcc = matthews_corrcoef(y_test, predictions)

    results.append([name, accuracy, auc, precision, recall, f1, mcc])
    joblib.dump(model, "model/" + file_names[name])  # Save the trained model.
    print("\n", name)
    print("Confusion matrix:\n", confusion_matrix(y_test, predictions))

# Display the results in one table.
columns = ["ML Model Name", "Accuracy", "AUC", "Precision", "Recall", "F1", "MCC"]
comparison = pd.DataFrame(results, columns=columns)
print(comparison.round(4).to_string(index=False))



 Logistic Regression
Confusion matrix:
 [[71  1]
 [ 3 39]]

 Decision Tree
Confusion matrix:
 [[70  2]
 [ 7 35]]

 KNN
Confusion matrix:
 [[71  1]
 [ 6 36]]

 Naive Bayes
Confusion matrix:
 [[72  0]
 [ 7 35]]

 Support Vector Machine
Confusion matrix:
 [[72  0]
 [ 3 39]]
         ML Model Name  Accuracy    AUC  Precision  Recall     F1    MCC
   Logistic Regression    0.9649 0.9960     0.9750  0.9286 0.9512 0.9245
         Decision Tree    0.9211 0.9448     0.9459  0.8333 0.8861 0.8299
                   KNN    0.9386 0.9825     0.9730  0.8571 0.9114 0.8688
           Naive Bayes    0.9386 0.9934     1.0000  0.8333 0.9091 0.8715
Support Vector Machine    0.9737 0.9947     1.0000  0.9286 0.9630 0.9442


In [49]:
# Select a winner using the highest F1 score.
best_row = comparison.loc[comparison["F1"].idxmax()]
print("Highest F1 model:", best_row["ML Model Name"])
print("F1 score:", round(best_row["F1"], 4))


Highest F1 model: Support Vector Machine
F1 score: 0.963
